In [37]:

from tomobase.tiltschemes import GRS
from tomobase import processes
from tomobase.data import Volume, Sinogram
from tomobase import processes
import numpy as np
import os
#import ssim
from skimage.metrics import structural_similarity as ssim
import copy
import stackview

folder_name= r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\Volumes128\Rod-D-4.0'


ts = GRS(-70, 70,0)
angles = np.array([ts.get_angle() for i in range(100)])


def subsection_sinogram(sino, start, end):
    sinogram = copy.deepcopy(sino)
    sinogram.data = sinogram.data[start:end,:, :]
    sinogram.angles = sinogram.angles[start:end]
    sinogram.times = sinogram.times[start:end]
    return sinogram

for i in range(1,101):
    vol_path = os.path.join(folder_name, f'{i}data.rec')
    vol = Volume.from_file(vol_path)
    sino_img = processes.project(vol, [angles[i-1]])
    sino_img.times = np.array([i])
    if i == 1:
        sino = copy.deepcopy(sino_img)
    else:
        sino.data = np.concatenate((sino.data, sino_img.data), axis=0)
        sino.angles = np.concatenate((sino.angles, sino_img.angles), axis=0)
        sino.times = np.concatenate((sino.times, sino_img.times), axis=0)

start=0
end=1
keep_going = True
windows = []
print('started', sino.data.shape)

sino_decremented = copy.deepcopy(sino)
sino_incremented = copy.deepcopy(sino)
while end <= sino.data.shape[0]:
    window = (start, end)

    sino_sub = subsection_sinogram(sino, start, end)
    rec = processes.astra_reconstruct(sino_sub, iterations=100)
    time =  np.mean(sino_sub.times)
    vol_path = os.path.join(folder_name, f'{int(round(time))}data.rec')
    vol = Volume.from_file(vol_path)
    ssim_val = ssim(rec.data, vol.data, data_range=vol.data.max()-vol.data.min())

    if end-start > 1:
        sino_decremented = subsection_sinogram(sino, start, end-1)
        time =  np.mean(sino_decremented.times)
        vol_path = os.path.join(folder_name, f'{int(round(time))}data.rec')
        vol = Volume.from_file(vol_path)
        rec_dec = processes.astra_reconstruct(sino_decremented, iterations=100)
        ssim_dec = ssim(rec_dec.data, vol.data, data_range=vol.data.max()-vol.data.min())
    else:
        sino_decremented.data = np.array([])
        sino_decremented.angles = np.array([])
        sino_decremented.times = np.array([])
        ssim_dec = 0
    
    if end >= sino.data.shape[0]:
        ssim_inc = 0
        sino_incremented.data = np.array([])
        sino_incremented.angles = np.array([])
        sino_incremented.times = np.array([])
    else:
        sino_incremented = subsection_sinogram(sino, start, end+1)
        time =  np.mean(sino_incremented.times)
        vol_path = os.path.join(folder_name, f'{int(round(time))}data.rec')
        vol = Volume.from_file(vol_path)
        rec_inc = processes.astra_reconstruct(sino_incremented, iterations=100)
        ssim_inc = ssim(rec_inc.data, vol.data, data_range=vol.data.max()-vol.data.min())

    print('-----------------------------------')
    print('-----------------------------------')
    print('vol_details:', vol.data.shape, vol.data.min(), vol.data.max())
    print('sino_details:', sino_sub.data.shape, sino_sub.angles, sino_sub.times)
    print('sino_dec_details:', sino_decremented.data.shape, sino_decremented.angles, sino_decremented.times)
    print('sino_inc_details:', sino_incremented.data.shape, sino_incremented.angles, sino_incremented.times)
    print(f'Window: {start}-{end}, SSIM: {ssim_val:.4f}, SSIM-: {ssim_dec:.4f}, SSIM+: {ssim_inc:.4f}')

    if ssim_inc > ssim_val:
        end += 1
    elif ssim_dec >= ssim_val:
        end -= 1
    else:
        windows.append((start, end))
        start+=1
        end +=1





2025-12-12 17:07:20,566 - DEBUG - ()
2025-12-12 17:07:20,567 - DEBUG - {'volume': <tomobase.data.volume.Volume object at 0x000001E835940090>, 'angles': [-70.0], 'use_gpu': True}
100%|██████████| 128/128 [00:00<00:00, 578.90it/s]
2025-12-12 17:07:20,937 - DEBUG - ()
2025-12-12 17:07:20,937 - DEBUG - {'volume': <tomobase.data.volume.Volume object at 0x000001E835942610>, 'angles': [16.52], 'use_gpu': True}
100%|██████████| 128/128 [00:00<00:00, 1044.80it/s]
2025-12-12 17:07:21,195 - DEBUG - ()
2025-12-12 17:07:21,195 - DEBUG - {'volume': <tomobase.data.volume.Volume object at 0x000001E835941C90>, 'angles': [-36.95], 'use_gpu': True}
100%|██████████| 128/128 [00:00<00:00, 1069.71it/s]
2025-12-12 17:07:21,451 - DEBUG - ()
2025-12-12 17:07:21,452 - DEBUG - {'volume': <tomobase.data.volume.Volume object at 0x000001E8358B7550>, 'angles': [49.57], 'use_gpu': True}
100%|██████████| 128/128 [00:00<00:00, 1081.16it/s]
2025-12-12 17:07:21,703 - DEBUG - ()
2025-12-12 17:07:21,703 - DEBUG - {'volume'

started (100, 128, 128)


2025-12-12 17:07:46,445 - DEBUG - ()
2025-12-12 17:07:46,447 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x000001E8358B2DD0>, 'method': 'sirt', 'iterations': 100, 'use_gpu': True}
2025-12-12 17:07:46,448 - INFO - Reconstructing...
2025-12-12 17:07:46,449 - INFO - Reconstruction using the SIRT_CUDA algorithm on the GPU...
100%|██████████| 128/128 [00:11<00:00, 11.45it/s]
2025-12-12 17:07:57,683 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>
2025-12-12 17:07:58,245 - DEBUG - ()
2025-12-12 17:07:58,245 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x000001E8358F6710>, 'method': 'sirt', 'iterations': 100, 'use_gpu': True}
2025-12-12 17:07:58,246 - INFO - Reconstructing...
2025-12-12 17:07:58,246 - INFO - Reconstruction using the SIRT_CUDA algorithm on the GPU...
100%|██████████| 128/128 [00:10<00:00, 12.17it/s]
2025-12-12 17:08:08,784 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>


-----------------------------------
-----------------------------------
vol_details: (128, 128, 128) 0.0 1.0
sino_details: (1, 128, 128) [-70.] [1]
sino_dec_details: (0,) [] []
sino_inc_details: (2, 128, 128) [-70.    16.52] [1 2]
Window: 0-1, SSIM: 0.6772, SSIM-: 0.0000, SSIM+: 0.8557


2025-12-12 17:08:09,152 - DEBUG - ()
2025-12-12 17:08:09,153 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x000001E8373790D0>, 'method': 'sirt', 'iterations': 100, 'use_gpu': True}
2025-12-12 17:08:09,153 - INFO - Reconstructing...
2025-12-12 17:08:09,154 - INFO - Reconstruction using the SIRT_CUDA algorithm on the GPU...
100%|██████████| 128/128 [00:10<00:00, 12.32it/s]
2025-12-12 17:08:19,583 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>
2025-12-12 17:08:20,129 - DEBUG - ()
2025-12-12 17:08:20,130 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x000001E83737B550>, 'method': 'sirt', 'iterations': 100, 'use_gpu': True}
2025-12-12 17:08:20,130 - INFO - Reconstructing...
2025-12-12 17:08:20,130 - INFO - Reconstruction using the SIRT_CUDA algorithm on the GPU...
100%|██████████| 128/128 [00:11<00:00, 11.63it/s]
2025-12-12 17:08:31,160 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>
2025-12-12 17:08:31,527 - DEBUG - ()
2025-12

...

-----------------------------------
-----------------------------------
vol_details: (128, 128, 128) 0.0 1.0
sino_details: (6, 128, 128) [-56.67  29.85 -23.62  62.9    9.43 -44.05] [ 95  96  97  98  99 100]
sino_dec_details: (5, 128, 128) [-56.67  29.85 -23.62  62.9    9.43] [95 96 97 98 99]
sino_inc_details: (0,) [] []
Window: 94-100, SSIM: 0.9589, SSIM-: 0.9612, SSIM+: 0.0000


2025-12-12 19:09:04,201 - DEBUG - ()
2025-12-12 19:09:04,202 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x000001E838FAB550>, 'method': 'sirt', 'iterations': 100, 'use_gpu': True}
2025-12-12 19:09:04,202 - INFO - Reconstructing...
2025-12-12 19:09:04,203 - INFO - Reconstruction using the SIRT_CUDA algorithm on the GPU...
100%|██████████| 128/128 [00:09<00:00, 13.22it/s]
2025-12-12 19:09:13,913 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>
2025-12-12 19:09:15,072 - DEBUG - ()
2025-12-12 19:09:15,072 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x000001E835905B10>, 'method': 'sirt', 'iterations': 100, 'use_gpu': True}
2025-12-12 19:09:15,073 - INFO - Reconstructing...
2025-12-12 19:09:15,073 - INFO - Reconstruction using the SIRT_CUDA algorithm on the GPU...
100%|██████████| 128/128 [00:09<00:00, 12.84it/s]
2025-12-12 19:09:25,063 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>
2025-12-12 19:09:26,209 - DEBUG - ()
2025-12

-----------------------------------
-----------------------------------
vol_details: (128, 128, 128) 0.0 1.0
sino_details: (5, 128, 128) [-56.67  29.85 -23.62  62.9    9.43] [95 96 97 98 99]
sino_dec_details: (4, 128, 128) [-56.67  29.85 -23.62  62.9 ] [95 96 97 98]
sino_inc_details: (6, 128, 128) [-56.67  29.85 -23.62  62.9    9.43 -44.05] [ 95  96  97  98  99 100]
Window: 94-99, SSIM: 0.9612, SSIM-: 0.9324, SSIM+: 0.9589


2025-12-12 19:09:36,312 - DEBUG - ()
2025-12-12 19:09:36,313 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x000001E835BD90D0>, 'method': 'sirt', 'iterations': 100, 'use_gpu': True}
2025-12-12 19:09:36,313 - INFO - Reconstructing...
2025-12-12 19:09:36,314 - INFO - Reconstruction using the SIRT_CUDA algorithm on the GPU...
100%|██████████| 128/128 [00:09<00:00, 13.19it/s]
2025-12-12 19:09:46,050 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>
2025-12-12 19:09:46,716 - DEBUG - ()
2025-12-12 19:09:46,716 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x000001E838FAB550>, 'method': 'sirt', 'iterations': 100, 'use_gpu': True}
2025-12-12 19:09:46,717 - INFO - Reconstructing...
2025-12-12 19:09:46,717 - INFO - Reconstruction using the SIRT_CUDA algorithm on the GPU...
100%|██████████| 128/128 [00:09<00:00, 13.21it/s]
2025-12-12 19:09:56,427 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>


-----------------------------------
-----------------------------------
vol_details: (128, 128, 128) 0.0 1.0
sino_details: (5, 128, 128) [ 29.85 -23.62  62.9    9.43 -44.05] [ 96  97  98  99 100]
sino_dec_details: (4, 128, 128) [ 29.85 -23.62  62.9    9.43] [96 97 98 99]
sino_inc_details: (0,) [] []
Window: 95-100, SSIM: 0.9557, SSIM-: 0.9498, SSIM+: 0.0000


In [38]:
import pandas as pd 

#save windos2
pd.DataFrame(windows).to_csv('windows_rodD4.0.csv', index=False, header=False)
print(len(windows))
for window in windows:
    print('Window selected:', window, 'size:', window[1]-window[0], 'time average:', np.mean(sino.times[window[0]:window[1]]))

96
Window selected: (0, 11) size: 11 time average: 6.0
Window selected: (1, 20) size: 19 time average: 11.0
Window selected: (2, 27) size: 25 time average: 15.0
Window selected: (3, 27) size: 24 time average: 15.5
Window selected: (4, 27) size: 23 time average: 16.0
Window selected: (5, 31) size: 26 time average: 18.5
Window selected: (6, 32) size: 26 time average: 19.5
Window selected: (7, 32) size: 25 time average: 20.0
Window selected: (8, 33) size: 25 time average: 21.0
Window selected: (9, 35) size: 26 time average: 22.5
Window selected: (10, 35) size: 25 time average: 23.0
Window selected: (11, 35) size: 24 time average: 23.5
Window selected: (12, 35) size: 23 time average: 24.0
Window selected: (13, 35) size: 22 time average: 24.5
Window selected: (14, 36) size: 22 time average: 25.5
Window selected: (15, 37) size: 22 time average: 26.5
Window selected: (16, 40) size: 24 time average: 28.5
Window selected: (17, 39) size: 22 time average: 28.5
Window selected: (18, 39) size: 21 t